# P300 BCI as a Communication Channel for Catatonic Patients
### Anastasia Zabolotna · Sigmoid · 2026

**Датасет:** BCI Competition III, Dataset II  
Зеркало: https://github.com/Manucar/p300-speller/tree/main/dataset  
Скачай: `Subject_A_Train.mat`, `Subject_B_Train.mat` (и Test-версии если нужны)

**Как запустить:**  
1. Загрузи `.mat` файлы через панель "Файлы" слева (иконка папки)  
2. `Runtime → Run all`  
3. Подожди ~5 мин при первом запуске (фильтрация EEG — самый долгий шаг), затем мгновенно из кэша

In [ ]:
!pip install -q numpy scipy scikit-learn matplotlib

In [ ]:
import os, json, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import scipy.io as sio
from scipy.signal import butter, filtfilt, decimate

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA

DATA_DIR = "/content"  # папка с .mat файлами (куда Colab кладёт загруженные)
FIGURES_DIR = "/content/figures"
RANDOM_SEED = 42
N_CV_FOLDS = 5


FS = 240# частота дискретизации, Гц
EPOCH_MS = 667  # длина эпохи, мс
EPOCH_SAMPLES = int(FS * EPOCH_MS / 1000)  # 160 отсчётов → 40 после децимации
DOWNSAMPLE = 4
CZ_IDX = 10 # канал ≈ Cz (система 10-20)

SPELLER = np.array([
    list("ABCDEF"), list("GHIJKL"), list("MNOPQR"),
    list("STUVWX"), list("YZ1234"), list("56789_"),
])

os.makedirs(FIGURES_DIR, exist_ok=True)
print("Всё импортировано. DATA_DIR =", DATA_DIR)

In [ ]:
required = [
    "Subject_A_Train.mat",
    "Subject_A_Test.mat",
    "Subject_B_Train.mat",
    "Subject_B_Test.mat",
]
all_ok = True
for fname in required:
    path = os.path.join(DATA_DIR, fname)
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1024 / 1024
        status = "OK" if mb > 50 else "WARNING: слишком мал, возможно не докачался"
        print(f"  {status:35s} {fname}  ({mb:.0f} МБ)")
        if mb < 50: all_ok = False
    else:
        print(f"  ОТСУТСТВУЕТ: {fname}")
        all_ok = False

if all_ok:
    print("\nВсе файлы найдены — можно запускать пайплайн.")
else:
    print("\n Загрузи недостающие файлы и перезапусти ячейку.")

 Препроцессинг и эпохинг

Для каждого trial (символа) применяем:
1. **Bandpass filter** Butterworth 4-го порядка, 0.1–20 Гц (zero-phase `filtfilt`)  
2. **Epoch extraction** — 667 мс окно после каждой вспышки (Flashing)  
3. **Decimation ×4** — 160 → 40 отсчётов (2560 признаков на эпоху)

Результат кэшируется в `.npz` — повторный запуск мгновенный.

In [ ]:
def _find_flash_onsets(fl, sc, st):
    diff = np.diff(np.concatenate([[0], fl]))
    idx  = np.where(diff == 1)[0]
    return idx, sc[idx].astype(int), st[idx].astype(int)


def extract_epochs(mat_path, use_cache=True):
    cache_path = mat_path.replace(".mat", "_epochs.npz")

    if use_cache and os.path.exists(cache_path):
        c = np.load(cache_path, allow_pickle=True)
        tc = c["target_chars"]
        print(f"  Loaded from cache: {os.path.basename(cache_path)}")
        return c["X"], c["y"], c["trial_idx"], c["codes"],                str(tc) if tc.ndim == 0 else tc

    if not os.path.exists(mat_path):
        raise FileNotFoundError(
            f"Файл не найден: {mat_path}\n"
            "Загрузи .mat файлы через панель 'Файлы' слева.")
    if os.path.getsize(mat_path) < 50 * 1024 * 1024:
        raise OSError(
            f"{os.path.basename(mat_path)} слишком мал — "
            "скорее всего файл скачался не полностью. Удали и загрузи заново.")

    data = sio.loadmat(mat_path)
    signal = data["Signal"]       # (n_trials, n_time, 64)
    flashing  = data["Flashing"]
    stim_code = data["StimulusCode"]
    stim_type = data.get("StimulusType")
    target_chars = data["TargetChar"][0] if "TargetChar" in data else None
    nyq  = FS / 2
    b, a = butter(4, [0.1 / nyq, 20.0 / nyq], btype="band")  # bandpass filter

    all_X, all_y, all_tr, all_co = [], [], [], []
    n_trials = signal.shape[0]
    print(f"  Processing {n_trials} trials from {os.path.basename(mat_path)} ...")

    for t in range(n_trials):
        sig  = signal[t].astype(np.float64)
        sig = filtfilt(b, a, sig, axis=0)   # zero-phase bandpass
        type_row = stim_type[t] if stim_type is not None                    else np.zeros(sig.shape[0])

        onsets, codes, types = _find_flash_onsets(
            flashing[t], stim_code[t], type_row)

        for onset, code, typ in zip(onsets, codes, types):
            epoch = sig[onset : onset + EPOCH_SAMPLES]  # extract epoch
            if epoch.shape[0] < EPOCH_SAMPLES:
                continue
            epoch = decimate(epoch, DOWNSAMPLE, axis=0, zero_phase=True)
            all_X.append(epoch); all_y.append(typ)
            all_tr.append(t);    all_co.append(code)

        if (t + 1) % 20 == 0:
            print(f"    {t+1}/{n_trials} trials done")

    X  = np.stack(all_X).astype(np.float32)
    y, trial_idx, codes = map(np.array, [all_y, all_tr, all_co])

    np.savez_compressed(cache_path, X=X, y=y, trial_idx=trial_idx,
                        codes=codes,
                        target_chars=target_chars if target_chars is not None else "")
    print(f"  Cache saved: {cache_path}")
    return X, y, trial_idx, codes, target_chars


print("Функции препроцессинга определены.")

In [ ]:
t0 = time.time()
print("[1/4] Preprocessing & epoch extraction")

XA, yA, trA, coA, tcA = extract_epochs(
    os.path.join(DATA_DIR, "Subject_A_Train.mat"))
XB, yB, trB, coB, tcB = extract_epochs(
    os.path.join(DATA_DIR, "Subject_B_Train.mat"))

print(f"\nSubject A: {XA.shape}  |  Subject B: {XB.shape}")
print(f"Class ratio A: target={yA.sum()}, non-target={len(yA)-yA.sum()} (1:{len(yA)//yA.sum()-1})")
print(f"Class ratio B: target={yB.sum()}, non-target={len(yB)-yB.sum()} (1:{len(yB)//yB.sum()-1})")
print(f"Done in {time.time()-t0:.1f}s")

 Figure 1 - Pipeline Diagram

In [ ]:
def plot_pipeline_diagram():
    from matplotlib.patches import FancyBboxPatch
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.set_xlim(0, 12); ax.set_ylim(0, 4); ax.axis("off")
    boxes = [
        (0.2,  1.3, 1.6, 1.4, "Raw EEG",        "64 ch, 240 Hz",         "#2E86AB"),
        (2.1,  1.3, 1.8, 1.4, "Preprocessing",  "0.1-20 Hz\nepoch 667ms","#A23B72"),
        (4.1,  1.3, 1.8, 1.4, "Feature Eng.",   "Flatten+z-score\n2560d","#F18F01"),
        (6.1,  1.3, 1.8, 1.4, "LDA Classifier", "shrinkage\nbinary P300","#C73E1D"),
        (8.1,  1.3, 1.8, 1.4, "Aggregation",    "avg scores\nrow/col",   "#3B1F2B"),
        (10.1, 1.3, 1.6, 1.4, "Symbol\nOutput","6x6 matrix",             "#2D6A4F"),
    ]
    for x, y, w, h, label, sub, color in boxes:
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                     facecolor=color, edgecolor="white", lw=2, alpha=0.92))
        ax.text(x+w/2, y+h/2+(0.18 if sub else 0), label,
                ha="center", va="center", fontsize=10, fontweight="bold", color="white")
        if sub:
            ax.text(x+w/2, y+h/2-0.28, sub, ha="center", va="center",
                    fontsize=7.5, color="#e8e8e8", style="italic")
    for x1, x2 in [(1.8,2.1),(3.9,4.1),(5.9,6.1),(7.9,8.1),(9.9,10.1)]:
        ax.annotate("", xy=(x2,2.0), xytext=(x1,2.0),
                    arrowprops=dict(arrowstyle="->", color="#555", lw=2))
    ax.annotate("", xy=(6.1,1.0), xytext=(4.1,1.0),
                arrowprops=dict(arrowstyle="<->", color="#F18F01", lw=1.5))
    ax.text(5.1, 0.75, "5-fold CV (trial-level)", ha="center",
            fontsize=8, color="#F18F01", style="italic")
    ax.set_title("P300 BCI Classification Pipeline", fontsize=13,
                 fontweight="bold", pad=10)
    fig.tight_layout()
    path = f"{FIGURES_DIR}/fig1_pipeline_diagram.png"
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)
    print(f"Saved: {path}")

plot_pipeline_diagram()

Figure 2 - ERP Curves (target vs non-target)

In [ ]:
def plot_erp(X, y, label):
    tgt    = X[y==1, :, CZ_IDX]
    nontgt = X[y==0, :, CZ_IDX]
    t_ms   = np.linspace(0, 667, X.shape[1])
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(t_ms, tgt.mean(0),    color="#d62728", lw=2, label="Target (P300 present)")
    ax.fill_between(t_ms, tgt.mean(0)-tgt.std(0)/tgt.shape[0]**0.5,
                          tgt.mean(0)+tgt.std(0)/tgt.shape[0]**0.5,
                    color="#d62728", alpha=0.2)
    ax.plot(t_ms, nontgt.mean(0), color="#1f77b4", lw=2, label="Non-target")
    ax.fill_between(t_ms, nontgt.mean(0)-nontgt.std(0)/nontgt.shape[0]**0.5,
                          nontgt.mean(0)+nontgt.std(0)/nontgt.shape[0]**0.5,
                    color="#1f77b4", alpha=0.2)
    ax.axvspan(250, 450, color="gray", alpha=0.15, label="P300 window (250-450 ms)")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_xlabel("Time after stimulus, ms"); ax.set_ylabel("Amplitude, μV")
    ax.set_title(f"Averaged ERP signal — Subject {label}"); ax.legend(fontsize=9)
    fig.tight_layout()
    path = f"{FIGURES_DIR}/fig2_erp_{label}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)
    print(f"Saved: {path}")

plot_erp(XA, yA, "A")
plot_erp(XB, yB, "B")

Figure 3 - PCA Projection of Feature Space

In [ ]:
def plot_pca(X, y, label):
    feat  = X.reshape(len(X), -1)
    feat2 = StandardScaler().fit_transform(feat)
    f2d = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(feat2)
    pca = PCA(n_components=2, random_state=RANDOM_SEED).fit(feat2)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(f2d[y==0,0], f2d[y==0,1], c="#1f77b4", alpha=0.2, s=5, label="Non-target")
    ax.scatter(f2d[y==1,0], f2d[y==1,1], c="#d62728", alpha=0.4, s=10, label="Target (P300)")
    ax.set_xlabel(f"PC1"); ax.set_ylabel("PC2")
    ax.set_title(f"PCA Projection — Subject {label}"); ax.legend()
    fig.tight_layout()
    path = f"{FIGURES_DIR}/fig3_pca_{label}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)
    print(f"Saved: {path}")

plot_pca(XA, yA, "A")

5-Fold Cross-Validation (Figure 4)

Разбиение по **символам** (trial-level), не по эпохам - иначе утечка данных.

LDA c shrinkage-регуляризацией (Ledoit-Wolf):  
`w* = argmax_w  [w^T S_B w] / [w^T S_W w]`

**Агрегация до символа:** усредняем score по repetitions для каждого из 12 кодов стимула, строка + столбец с max score - декодируем букву из матрицы 6×6.

In [ ]:
def epochs_to_features(X):
    return X.reshape(len(X), -1)


def aggregate_to_char(scores, trial_idx, codes, target_chars, trials, n_rep=None):

    correct = total = 0
    for t in trials:
        m  = trial_idx == t
        ms, mc = scores[m], codes[m]
        if n_rep is not None:
            cut = min(n_rep, len(ms) // 12) * 12
            ms, mc = ms[:cut], mc[:cut]

        scores_per_code = np.zeros(13)
        for c in range(1, 13):
            mask = mc == c
            if mask.sum() > 0:
                scores_per_code[c] = ms[mask].mean()
        col  = np.argmax(scores_per_code[1:7])  + 1
        row  = np.argmax(scores_per_code[7:13]) + 1
        pred = SPELLER[row - 1, col - 1]

        if target_chars is not None:
            total += 1; correct += (pred == target_chars[t])
    return correct / total if total > 0 else 0.0


def run_5fold_cv(X, y, trial_idx, codes, target_chars, label=""):
    feat  = epochs_to_features(X)
    unique_trials = np.unique(trial_idx)
    kf = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    aucs, accs, chars = [], [], []

    print(f"\n  {label} — 5-fold CV (trial-level):")
    for fold, (tr_idx, te_idx) in enumerate(kf.split(unique_trials)):
        train_tr   = set(unique_trials[tr_idx])
        test_tr    = sorted(unique_trials[te_idx])
        train_mask = np.isin(trial_idx, list(train_tr))
        test_mask  = np.isin(trial_idx, test_tr)

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(feat[train_mask])
        X_all  = scaler.transform(feat)

        clf = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
        clf.fit(X_tr, y[train_mask])

        sc_test = clf.decision_function(X_all[test_mask])
        auc = roc_auc_score(y[test_mask], sc_test)
        acc = accuracy_score(y[test_mask], clf.predict(X_all[test_mask]))
        sc_all  = clf.decision_function(X_all)
        char_ac = aggregate_to_char(sc_all, trial_idx, codes, target_chars, test_tr)

        aucs.append(auc); accs.append(acc); chars.append(char_ac)
        print(f"    Fold {fold+1}: AUC={auc:.3f}  epoch_acc={acc:.3f}  char_acc={char_ac:.3f}")

    a, b, c = np.array(aucs), np.array(accs), np.array(chars)
    print(f"  AUC: {a.mean():.3f} ± {a.std():.3f}")
    print(f"  Epoch acc: {b.mean():.3f} ± {b.std():.3f}")
    print(f"  Char acc : {c.mean():.3f} ± {c.std():.3f}")
    return a, b, c


print("[2/4] 5-fold Cross-Validation")
aucs_A, accs_A, chars_A = run_5fold_cv(XA, yA, trA, coA, tcA, "Subject A")
aucs_B, accs_B, chars_B = run_5fold_cv(XB, yB, trB, coB, tcB, "Subject B")

In [ ]:
def plot_cv_results(aucs_A, chars_A, aucs_B, chars_B):
    subjects = ["Subject A", "Subject B"]
    char_m = [chars_A.mean(), chars_B.mean()]
    char_s = [chars_A.std(),  chars_B.std()]
    auc_m= [aucs_A.mean(),  aucs_B.mean()]
    auc_s = [aucs_A.std(),   aucs_B.std()]
    x, w = np.arange(2), 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    b1 = ax.bar(x-w/2, [v*100 for v in char_m], w,
                yerr=[v*100 for v in char_s], capsize=6,
                label="Character accuracy, %", color="#2ca02c", error_kw={"lw":2})
    b2 = ax.bar(x+w/2, [v*100 for v in auc_m], w,
                yerr=[v*100 for v in auc_s], capsize=6,
                label="Epoch AUC, %", color="#ff7f0e", error_kw={"lw":2})
    ax.set_xticks(x); ax.set_xticklabels(subjects)
    ax.set_ylabel("%"); ax.set_ylim(0, 115)
    ax.set_title("5-fold CV Results — LDA (mean ± std)"); ax.legend()
    for bars, means, stds in [(b1,char_m,char_s),(b2,auc_m,auc_s)]:
        for bar, m, s in zip(bars, means, stds):
            h = bar.get_height()
            ax.annotate(f"{m*100:.1f}\n±{s*100:.1f}",
                        xy=(bar.get_x()+bar.get_width()/2, h),
                        xytext=(0,5), textcoords="offset points",
                        ha="center", fontsize=8)
    fig.tight_layout()
    path = f"{FIGURES_DIR}/fig4_cv_results.png"
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)
    print(f"Saved: {path}")

plot_cv_results(aucs_A, chars_A, aucs_B, chars_B)

Figure 5 - Speed-Accuracy Trade-off (5-fold CV)

Как character accuracy меняется при использовании только первых **N повторений** (N = 1-15)?  
Больше повторений → выше точность, но медленнее спеллер.  
5-fold CV → mean ± std по фолдам.

In [ ]:
def repetition_cv_analysis(X, y, trial_idx, codes, target_chars, label):
    feat = epochs_to_features(X)
    unique_trials = np.unique(trial_idx)
    kf = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    per_fold = []
    print(f"  {label} repetition CV ...")
    for tr_idx, te_idx in kf.split(unique_trials):
        train_tr  = set(unique_trials[tr_idx])
        test_tr = sorted(unique_trials[te_idx])
        train_mask = np.isin(trial_idx, list(train_tr))
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(feat[train_mask])
        X_all = scaler.transform(feat)
        clf = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
        clf.fit(X_tr, y[train_mask])
        sc_all = clf.decision_function(X_all)
        fold_accs = [aggregate_to_char(sc_all, trial_idx, codes, target_chars,
                                       test_tr, n_rep=n)
                     for n in range(1, 16)]
        per_fold.append(fold_accs)
    per_fold = np.array(per_fold)
    return per_fold.mean(axis=0), per_fold.std(axis=0)


print("[3/4] Speed-accuracy trade-off (5-fold CV)")
mA, sA = repetition_cv_analysis(XA, yA, trA, coA, tcA, "Subject A")
mB, sB = repetition_cv_analysis(XB, yB, trB, coB, tcB, "Subject B")

In [ ]:
def plot_repetitions(mA, sA, mB, sB):
    reps = np.arange(1, 16)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(reps, mA*100, marker="o", color="#d62728", label="Subject A", lw=2)
    ax.fill_between(reps, (mA-sA)*100, (mA+sA)*100, color="#d62728", alpha=0.2)
    ax.plot(reps, mB*100, marker="s", color="#1f77b4", label="Subject B", lw=2)
    ax.fill_between(reps, (mB-sB)*100, (mB+sB)*100, color="#1f77b4", alpha=0.2)
    ax.set_xlabel("Number of repetitions"); ax.set_ylabel("Character accuracy, %")
    ax.set_title("Speed-Accuracy Trade-off (5-fold CV, mean ± std)", fontsize=10.5)
    ax.set_xticks(reps); ax.set_ylim(0, 115); ax.legend()
    fig.tight_layout()
    path = f"{FIGURES_DIR}/fig5_repetitions_cv.png"
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)
    print(f"Saved: {path}")

plot_repetitions(mA, sA, mB, sB)

Итоговая таблица результатов

In [ ]:
print("[4/4] Results summary")
print("RESULTS SUMMARY — 5-fold CV, LDA with shrinkage (Ledoit-Wolf)")
print(f"{'Subject':<12}  {'Epoch AUC':>20}  {'Epoch Acc':>20}  {'Char Acc':>20}")
for label, a, b_, c in [("Subject A", aucs_A, accs_A, chars_A),
                          ("Subject B", aucs_B, accs_B, chars_B)]:
    print(f"{label:<12}  {a.mean():.3f} ± {a.std():.3f}       "
          f"{b_.mean():.3f} ± {b_.std():.3f}       "
          f"{c.mean():.3f} ± {c.std():.3f}")
print("=" * 65)

# Сохраняем числа
results = {
    "subject_A": {
        "auc_mean": float(aucs_A.mean()), "auc_std": float(aucs_A.std()),
        "acc_mean": float(accs_A.mean()), "acc_std": float(accs_A.std()),
        "char_mean":float(chars_A.mean()),"char_std":float(chars_A.std()),
    },
    "subject_B": {
        "auc_mean": float(aucs_B.mean()), "auc_std": float(aucs_B.std()),
        "acc_mean": float(accs_B.mean()), "acc_std": float(accs_B.std()),
        "char_mean":float(chars_B.mean()),"char_std":float(chars_B.std()),
    },
    "repetition_A": {"mean": mA.tolist(), "std": sA.tolist()},
    "repetition_B": {"mean": mB.tolist(), "std": sB.tolist()},
}
rpath = f"{FIGURES_DIR}/results.json"
with open(rpath, "w") as f: json.dump(results, f, indent=2)
print(f"\nЧисловые результаты: {rpath}")
print(f"Графики: {FIGURES_DIR}/")